In [1]:
#%conda install numpy -c conda-forge
#%conda install ipykernel -c conda-forge

In [38]:
import numpy as np
import re
import json
from IPython.display import HTML
from IPython.display import IFrame
import hashlib

In [39]:
def read_orca_trj(filename, build_xyz_frames=True, convert_energy=True):
    # Parse an ORCA trajectory file (XYZ format).
    # This function was rewritten using DeepSeek.

    with open(filename, 'r') as f:
        lines = f.readlines()

    i = 0
    frames = []
    symbols = []
    energies_eh = []

    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        try:
            natoms = int(lines[i].strip())
        except ValueError:
            i += 1
            continue
        i += 1  # comment line
        comment = lines[i].strip()
        match = re.search(r'E\s+([-+]?\d*\.\d+|\d+)', comment)
        energies_eh.append(float(match.group(1)) if match else np.nan)
        i += 1  # atom lines
        coords = []
        sym = []
        for _ in range(natoms):
            parts = lines[i].split()
            if len(parts) >= 4:
                sym.append(parts[0])
                coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
            i += 1
        frames.append(coords)
        if not symbols:
            symbols = sym
        else:
            if sym != symbols:
                print("Warning: atom symbols changed between frames; using first frame's symbols.")

    coords = np.array(frames)
    energies_eh = np.array(energies_eh)
    n_steps, n_atoms, _ = coords.shape

    result = {
        'symbols': symbols,
        'coords': coords,
        'energies_eh': energies_eh,
        'n_steps': n_steps,
        'n_atoms': n_atoms
    }

    if build_xyz_frames:
        xyz_frames = []
        for i in range(n_steps):
            frame_str = f"{n_atoms}\nStep {i}  E {energies_eh[i]:.12f}\n"
            for j in range(n_atoms):
                frame_str += f"{symbols[j]} {coords[i,j,0]:.6f} {coords[i,j,1]:.6f} {coords[i,j,2]:.6f}\n"
            xyz_frames.append(frame_str)
        result['xyz_frames'] = xyz_frames

    if convert_energy:
        energies_kcal = (energies_eh - energies_eh[0]) * 627.509
        result['energies_kcal'] = energies_kcal

    return result

In [52]:
def render_trajectory_energy(xyz_frames, energies,
                             viewer_width=400, viewer_height=400,
                             energy_width=400, energy_height=400,
                             gap=20,                     
                             preserve_camera=False, 
                             filename=None):
    
    # this HTML thing was written by DeepSeek and reviewed by Felippe
    # nglview and py3Dmol did not work or could not do what I wanted...

    n_steps = len(xyz_frames)
    if hasattr(energies, 'tolist'):
        energy_list = energies.tolist()
    else:
        energy_list = list(energies)

    total_width = viewer_width + gap + energy_width
    container_height = max(viewer_height, energy_height)

    html_template = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <script src="https://3Dmol.csb.pitt.edu/build/3Dmol-min.js"></script>
        <script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
        <style>
            body { margin:0; padding:0; }
            #container {
                position: relative;
                width: %(total_width)dpx;
                height: %(container_height)dpx;
                margin: 0 auto;
            }
            #viewer {
                position: absolute;
                top: 0;
                left: 0;
                width: %(viewer_width)dpx;
                height: %(viewer_height)dpx;
                margin: 0;
                padding: 0;
                border: none;
                overflow: hidden;
            }
            #energyPlot {
                position: absolute;
                top: 0;
                left: %(energy_left)dpx;   /* viewer_width + gap */
                width: %(energy_width)dpx;
                height: %(energy_height)dpx;
                margin: 0;
                padding: 0;
                border: none;
                overflow: hidden;
            }
            #slider {
                display: block;
                width: %(total_width)dpx;
                margin: 10px auto 0;
            }
        </style>
    </head>
    <body>
    <div id="container">
        <div id="viewer"></div>
        <div id="energyPlot"></div>
    </div>
    <div style="text-align:center;">
        <input type="range" id="frameSlider" min="0" max="%(max_frame)d" value="0" step="1" style="width: %(total_width)dpx;">
    </div>

    <script>
    var frames = %(frames_json)s;
    var energies = %(energies_json)s;
    var preserve = %(preserve_camera)s;

    var viewer = $3Dmol.createViewer("viewer", {backgroundColor: "white"});
    viewer.addModel(frames[0], "xyz");
    viewer.setStyle({}, {stick:{}, sphere:{scale:0.3}});
    viewer.zoomTo();
    viewer.render();

    var energyDiv = document.getElementById('energyPlot');
    var energyTrace = {
        x: Array.from({length: energies.length}, (_, i) => i),
        y: energies,
        mode: 'lines+markers',
        line: {color: 'blue', width: 2},
        marker: {size: 6}
    };
    var vlineTrace = {
        x: [0, 0],
        y: [Math.min(...energies)-1, Math.max(...energies)+1],
        mode: 'lines',
        line: {color: 'red', dash: 'dash', width: 2},
        showlegend: false
    };
    var layout = {
        title: '',   // no title -> no top margin
        xaxis: {title: 'Step'},
        yaxis: {title: 'Relative energy (kcal/mol)'},
        width: %(energy_width)d,
        height: %(energy_height)d,
        margin: {l: 60, r: 20, t: 0, b: 40},
        showlegend: false
    };
    Plotly.newPlot(energyDiv, [energyTrace, vlineTrace], layout);

    document.getElementById('frameSlider').addEventListener('input', function() {
        var idx = parseInt(this.value);
        var saved_view = null;
        if (preserve) {
            saved_view = viewer.getView();
        }
        viewer.removeAllModels();
        viewer.addModel(frames[idx], "xyz");
        viewer.setStyle({}, {stick:{}, sphere:{scale:0.3}});
        viewer.zoomTo();
        if (saved_view !== null) {
            viewer.setView(saved_view);
        }
        viewer.render();
        Plotly.restyle(energyDiv, {'x': [[idx, idx]]}, [1]);
    });
    </script>
    </body>
    </html>
    """

    energy_left = viewer_width + gap

    html_code = html_template % {
        'viewer_width': viewer_width,
        'viewer_height': viewer_height,
        'energy_width': energy_width,
        'energy_height': energy_height,
        'energy_left': energy_left,
        'total_width': total_width,
        'container_height': container_height,
        'max_frame': n_steps - 1,
        'frames_json': json.dumps(xyz_frames),
        'energies_json': json.dumps(energy_list),
        'preserve_camera': 'true' if preserve_camera else 'false'
    }

    # Determine a unique filename if not provided
    if filename is None:
        hash_string = hashlib.md5(html_code.encode()).hexdigest()[:8]
        filename = f"viewer_{hash_string}.html"

    # Write to file
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_code)

    # Return IFrame pointing to the file
    return IFrame(src=filename, width=total_width + 20, height=container_height + 80)

In [53]:
path = "."

# 1. RELAXED SCAN

In [65]:
filename = path + "/00_relaxed_scan/scan_final.xyz"

data = read_orca_trj(filename)
print(f"Steps: {data['n_steps']}, Atoms: {data['n_atoms']}")

html_output = render_trajectory_energy(data['xyz_frames'], data['energies_kcal'], filename="view_relaxedscan.html")
display(html_output)

Steps: 26, Atoms: 9


# 1.1. Extract TS structure

In [55]:
frame_index = 17
xyz_string = data['xyz_frames'][frame_index]   

with open(f"01_TS_optimization/ts_guess.xyz", "w") as f:
    f.write(xyz_string)

# 2. TS OPTIMIZATION

In [66]:
filename = path + "/01_TS_optimization/opt_ts_trj.xyz"

data = read_orca_trj(filename)
print(f"Steps: {data['n_steps']}, Atoms: {data['n_atoms']}")

html_output = render_trajectory_energy(data['xyz_frames'], data['energies_kcal'], filename="view_TS_opt.html")
display(html_output)

Steps: 18, Atoms: 9


# 2.1. TS VIBRATIONAL ANALYSIS

In [67]:
filename = path + "/01_TS_optimization/freq_ts.hess.v006.xyz"

data = read_orca_trj(filename)
print(f"Steps: {data['n_steps']}, Atoms: {data['n_atoms']}")

html_output = render_trajectory_energy(data['xyz_frames'], data['energies_kcal'], filename="view_TS_freq.html")
display(html_output)

Steps: 20, Atoms: 9


# 3. IRC - INTRINSIC REACTION COORDINATE

In [68]:
filename = path + "/02_IRC/irc_IRC_Full_trj.xyz"

data = read_orca_trj(filename)
print(f"Steps: {data['n_steps']}, Atoms: {data['n_atoms']}")

html_output = render_trajectory_energy(data['xyz_frames'], data['energies_kcal'], filename="view_irc.html")
display(html_output)

Steps: 56, Atoms: 9


# 3.1. Extract frames for R and P

In [62]:
frame_index = 0
xyz_string = data['xyz_frames'][frame_index]   

with open(f"02_IRC/P.xyz", "w") as f:
    f.write(xyz_string)

frame_index = 55
xyz_string = data['xyz_frames'][frame_index]   

with open(f"02_IRC/R.xyz", "w") as f:
    f.write(xyz_string)

# 4.1. R OPTIMIZATION

In [63]:
filename = path + "/03_R-P_optimization/opt_R_trj.xyz"

data = read_orca_trj(filename)
print(f"Steps: {data['n_steps']}, Atoms: {data['n_atoms']}")

html_output = render_trajectory_energy(data['xyz_frames'], data['energies_kcal'], filename="view_R_opt.html")
display(html_output)

Steps: 14, Atoms: 9


# 4.2. P OPTIMIZATION

In [64]:
filename = path + "/03_R-P_optimization/opt_P_trj.xyz"

data = read_orca_trj(filename)
print(f"Steps: {data['n_steps']}, Atoms: {data['n_atoms']}")

html_output = render_trajectory_energy(data['xyz_frames'], data['energies_kcal'], filename="view_P_opt.html")
display(html_output)

Steps: 109, Atoms: 9
